# 文本分类训练和评估示例

这个notebook展示了如何使用我们的系统进行文本分类模型的训练和评估。

## 1. 导入必要的库

In [ ]:
# 导入notebook友好的训练工具
from classifier.notebook_utils import (
    NotebookTrainingConfig, 
    start_training, 
    quick_train,
    create_config,
    train
)

# 导入模型评估工具
from classifier.model_evaluator import ModelEvaluator, load_model

# 导入测试工具
from test_sqlite_loader import run_all_tests

import torch
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU数量: {torch.cuda.device_count()}")
    print(f"当前GPU: {torch.cuda.current_device()}")


## 2. 快速训练

In [ ]:
# 快速训练 - 测试模式（小规模、快速验证）
print("🚀 开始快速训练...")

best_model_path, run_dir = quick_train(
    batch_size=8,
    num_epochs=2,
    learning_rate=2e-4,
    use_lora=True,
    test_mode=True  # 启用测试模式
)

print(f"\n✅ 训练完成！")
print(f"📁 运行目录: {run_dir}")
print(f"🎯 最佳模型: {best_model_path}")


## 3. 自定义配置训练

In [ ]:
# 创建配置
config = create_config()

# 链式配置（每个方法都返回self，支持链式调用）
config.quick_setup(
    batch_size=16,
    num_epochs=6,
    learning_rate=2e-4,
    max_length=256,
    use_lora=True
).set_lora_config(
    r=32,  # 降低LoRA rank以减少计算量
    alpha=16,
    dropout=0.3
).set_data_split(
    train=0.8,
    valid=0.1,
    test=0.1
).auto_mode()  # 自动选择设备


In [ ]:
# 如果需要，可以开始训练（取消注释下面的代码）
# print("🚀 开始自定义配置训练...")
# best_model_path, run_dir = train(config)
# print(f"\n✅ 训练完成！")
# print(f"📁 运行目录: {run_dir}")
# print(f"🎯 最佳模型: {best_model_path}")

print("配置已创建，如需训练请取消注释上面的代码")

## 4. 模型加载和评估

In [ ]:
# 加载模型示例（使用上面训练的模型路径，或者指定已有模型路径）
# model_path = best_model_path  # 使用刚才训练的模型
model_path = "models/best_model"  # 或者指定已有模型路径

# 方法1：使用便捷函数
try:
    evaluator = load_model(model_path, device="auto")
    print("✅ 模型加载成功")
except Exception as e:
    print(f"❌ 模型加载失败: {e}")
    print("请确保模型路径正确，或先运行训练代码")


## 5. 单条文本预测

In [ ]:
# 单条预测示例
test_texts = [
    "这个产品质量很好，非常满意",
    "服务态度差，等了很久",
    "价格合理，性价比不错",
    "物流很快，包装完好"
]

# 如果模型加载成功，进行预测
if 'evaluator' in locals():
    print("🔍 单条文本预测:")
    print("=" * 50)
    
    for text in test_texts:
        # 简单预测
        labels = evaluator.predict_single(text, threshold=0.5)
        print(f"文本: {text}")
        print(f"预测标签: {labels}")
        
        # 详细预测（包含概率）
        detailed = evaluator.predict_single(text, threshold=0.3, return_probabilities=True)
        print(f"详细结果: {detailed['predicted_labels']}")
        
        # 显示前3个最高分数的标签
        sorted_scores = sorted(detailed['all_scores'].items(), key=lambda x: x[1], reverse=True)
        print(f"前3个分数: {sorted_scores[:3]}")
        print("-" * 50)
else:
    print("请先加载模型")


## 6. 批量预测

In [ ]:
# 创建结果DataFrame
import pandas as pd

# 批量预测
if 'evaluator' in locals():
    print("📊 批量预测:")
    
    batch_results = evaluator.predict_batch(
        texts=test_texts,
        threshold=0.5,
        batch_size=2,
        show_progress=True
    )
    
    print("\n批量预测结果:")
    for text, labels in zip(test_texts, batch_results):
        print(f"{text} -> {labels}")
        
    results_data = []
    for text, labels in zip(test_texts, batch_results):
        results_data.append({
            'text': text,
            'predicted_labels': ', '.join(labels),
            'num_labels': len(labels)
        })
    
    results_df = pd.DataFrame(results_data)
    print("\n📊 预测结果汇总:")
    print(results_df)
else:
    print("请先加载模型")
